# Tanager Mangrove Mapping - 04 Transferability

| | |
|---|---|
| **Authors**     | Muhammad Wahyu Ramadhan, Athar Abdurrahman B., Diniyarti |
| **Competition** | Planet Tanager Open Data Competition 2026 |
| **Topic**       | Transferable Mangrove Extent and Biomass Mapping Using Adaptive Spectral Thresholds |
| **Date**        | June 2026 |

---

**Scope:** Apply pipeline trained on Sangatta to Gujarat, El Salvador, Belize (+ HCMC if confirmed). Per-scene adaptive threshold recalibration — no retraining. Accuracy evaluation against GMW v3.

## 0. Environment Setup

In [ ]:
# Install dependencies (commented out for production)
# !pip install scikit-learn geopandas rasterio matplotlib joblib

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio

# ============================================================
# Project root
# ============================================================
ROOT        = Path("..").resolve()
DATA_PROC   = ROOT / "data" / "processed"
DATA_GMW    = ROOT / "data" / "gmw_v3"
OUT_MODELS  = ROOT / "outputs" / "models"
OUT_RESULTS = ROOT / "outputs" / "results"
OUT_FIGURES = ROOT / "outputs" / "figures"

sys.path.insert(0, str(ROOT))
from src.preprocessing import load_geotiff_bands, compute_all_indices
from src.classification import load_model
from src.transferability import run_all_transfer_sites, TRANSFER_SITES

TRAIN_SCENE_ID = "20250302_030003_92_4001"   # Sangatta

print(f"ROOT            : {ROOT}")
print(f"Training scene  : {TRAIN_SCENE_ID}")
print(f"Transfer sites  : {list(TRANSFER_SITES.values())}")

## 1. Load Trained Model

In [ ]:
# ============================================================
# RF model trained on Sangatta — used as-is for all sites
# ============================================================
rf_model = load_model(str(OUT_MODELS / f"rf_{TRAIN_SCENE_ID}.joblib"))
print(f"Model loaded     : rf_{TRAIN_SCENE_ID}.joblib")

## 2. Per-Site Transfer Pipeline

In [ ]:
# ============================================================
# Run transfer pipeline across all sites.
# run_all_transfer_sites() handles: band loading, indices,
# water masking, adaptive thresholds, RF prediction, raster save,
# and per-scene threshold JSON to outputs/results/.
# ============================================================
summary_df = run_all_transfer_sites(
    processed_dir = str(DATA_PROC),
    model_path    = str(OUT_MODELS / f"rf_{TRAIN_SCENE_ID}.joblib"),
    output_dir    = str(DATA_PROC),
    results_dir   = str(OUT_RESULTS),
)

summary_df.to_csv(OUT_RESULTS / "transferability_summary.csv", index=False)

# ============================================================
# Rebuild per-site results dict for downstream cells
# (visualization in cell 11, accuracy eval in cell 9).
# ============================================================
results = {}
for _, row in summary_df.iterrows():
    scene_id = row["scene_id"]
    site_key = row["site_name"].lower().replace(", ", "_").replace(" ", "_")
    data     = load_geotiff_bands(str(DATA_PROC), scene_id)
    indices  = compute_all_indices(data)

    # Load the saved extent map
    extent_path = DATA_PROC / f"{scene_id}_mangrove_extent.tif"
    with rasterio.open(extent_path) as src:
        extent_map = src.read(1)

    results[site_key] = {
        "scene_id"  : scene_id,
        "extent_map": extent_map,
        "indices"   : indices,
        "data"      : data,
    }

print(f"\nResults ready for {list(results.keys())}")

## 3. Accuracy Evaluation vs GMW v3

In [ ]:
# ============================================================
# Compare RF extent map against GMW v3 per site
# Metrics: OA, F1, IoU
# TODO: implement after GMW v3 subsets prepared by Dini
# ============================================================
accuracy_records = []

for site, res in results.items():
    gmw_path = DATA_GMW / f"gmw_{site}.geojson"
    if not gmw_path.exists():
        print(f"  {site:<12}: GMW v3 not found — skip")
        continue
    # TODO: rasterize GMW v3 to same grid, compute metrics
    print(f"  {site:<12}: TODO")

if accuracy_records:
    acc_df = pd.DataFrame(accuracy_records)
    acc_df.to_csv(OUT_RESULTS / "transferability_accuracy.csv", index=False)
    print(acc_df)

## 4. Visualization

In [ ]:
# ============================================================
# Multi-site extent map grid
# ============================================================
n_sites = len(results)
fig, axes = plt.subplots(1, n_sites, figsize=(5 * n_sites, 5))

if n_sites == 1:
    axes = [axes]

for ax, (site, res) in zip(axes, results.items()):
    ax.imshow(res["extent_map"] == 1, cmap="Greens")
    ax.set_title(site.capitalize())
    ax.axis("off")

plt.suptitle("Mangrove Extent — Transfer Sites (RF, no retraining)", y=1.02)
plt.tight_layout()
plt.savefig(OUT_FIGURES / "transferability_extent_maps.png", dpi=150, bbox_inches="tight")
plt.show()